# PA2 — Kaggle notebook: Task 1 setup check + Task 7 judging

Training and generation run locally (see `notebooks/PA2_local_pipeline.ipynb`). This notebook covers the two parts that need Kaggle:

- **Part A (Task 1):** run with **Accelerator = GPU T4 x2**. Clones the repo, installs dependencies without reinstalling PyTorch, confirms two T4s, prints versions, and runs the adapter tests.
- **Part B (Task 7):** run inside a **Kaggle Benchmarks task notebook** (Kaggle Benchmarks → "Create task"; requires a phone-verified account). Only task notebooks have `kaggle_benchmarks` and free model access to `google/gemini-3.6-flash`. Paste this notebook's cells in, set `RUN_PART_A_SETUP = False`, and run. It judges the saved pairwise JSONL files only; nothing is regenerated here.
  - Batching: 5 pairs per request, at least 5 s sleep between attempts, up to 3 retries.
  - Durability: checkpoints after each batch and keeps a failure log.
  - Resume: rerunning skips pairs already judged.

Everything is written under `/kaggle/working`. Save a notebook version when it finishes. To resume after a disconnect, attach the previous version's output as an input dataset; the copy cell restores the existing judgments first.

In [1]:
RUN_PART_A_SETUP = True      # Task 1 (needs GPU T4 x2)
RUN_PART_B_JUDGING = False    # Task 7 (needs Kaggle model access)
REPO_URL = "https://github.com/AbdullahMushtaq27100324/GenAI_PA2.git"
# Private repo: add a GitHub token (read-only "Contents" access to this repo) as a Kaggle secret named
# GITHUB_TOKEN via Add-ons -> Secrets. It is read at runtime and never written into the notebook.
JUDGE_MODEL = "google/gemini-3.6-flash"
PREVIOUS_JUDGMENTS = None    # e.g. "/kaggle/input/pa2-judge-v1/judgments" to resume from an earlier version

In [2]:
import os, sys, json, shutil, subprocess, zipfile
from pathlib import Path
# On Kaggle everything goes to /kaggle/working. Elsewhere (local test runs) use PA2_KAGGLE_WORK or a temp dir.
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(os.environ.get("PA2_KAGGLE_WORK", "/tmp/pa2_kaggle_work"))
WORK.mkdir(parents=True, exist_ok=True)
REPO = WORK / "pa2"
if not REPO.exists():
    url = REPO_URL
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        url = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
    except Exception:
        print("no GITHUB_TOKEN secret - cloning without credentials (works for public repos only)")
    res = subprocess.run(["git", "clone", "--depth", "1", url, str(REPO)], capture_output=True, text=True)
    assert res.returncode == 0, "git clone failed (check REPO_URL and the GITHUB_TOKEN secret)"
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print("repo at", REPO, "| commit", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())

repo at /kaggle/working/pa2 | commit 57a19af


## Part A — Task 1: environment and GPU check

In [3]:
if RUN_PART_A_SETUP:
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
    # Manual Section 5.2: keep Kaggle's CUDA PyTorch, avoid the stale torchao / PEFT conflict.
    subprocess.run("python -m pip uninstall -y -q torchao || true", shell=True)
    subprocess.run('python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub '
                   'accelerate tqdm pyyaml sentencepiece safetensors pytest', shell=True, check=True)
    subprocess.run("python -m pip install -q -e . --no-deps", shell=True, check=True)

Mon Oct 12 09:05:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   59C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [4]:
if RUN_PART_A_SETUP:
    import torch, transformers, datasets
    print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__)
    n = torch.cuda.device_count()
    for i in range(n):
        p = torch.cuda.get_device_properties(i)
        print(f"cuda:{i} {p.name} {p.total_memory / 2**30:.1f} GiB")
    assert n == 2 and all("T4" in torch.cuda.get_device_name(i) for i in range(n)), "expected two Tesla T4 GPUs"
    res = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_public_adapter_smoke.py", "tests/test_own_alignment.py"],
                         capture_output=True, text=True, env={**os.environ, "PYTHONPATH": str(REPO)})
    print(res.stdout[-3000:], res.stderr[-1000:])

torch 2.11.0+cu128 | transformers 4.57.6 | datasets 4.8.5
cuda:0 Tesla T4 14.6 GiB
cuda:1 Tesla T4 14.6 GiB
........................                                                 [100%]
=============================== warnings summary ===============================
tests/test_own_alignment.py::test_hf_hidden_path_matches_full_logits_path_values_and_grads
  /usr/local/lib/python3.13/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
    prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))

tests/test_own_alignment.py::test_hf_hidden_path_matches_full_logits_path_values_and_grads
tests/test_own_alignment.py::test_hf_hidden_path_matches_full_logits_path_values_and_grads
tests/test_own_alignment.py::test_hf_hidden_path_matches_full_logits_path_values_and_grads
tests/test_own_alignment.py::test_hf_hidden_path_matches_full_logits_path_values_and_grads
tests/test_own_alignment.py::test_hf_hidden_pa

## Part B — Task 7: pairwise judging
Pair files are read from `outputs/eval/*_pairs.jsonl` in the repo. They are committed by the local pipeline, or can come from an attached `kaggle_judge_bundle.zip` dataset.

In [5]:
if RUN_PART_B_JUDGING:
    if JUDGE_MODEL != "mock":  # "mock" = offline fake judge, only for testing this notebook outside Kaggle
        import kaggle_benchmarks as kbench
        print("available judge models:", sorted(kbench.llms)[:50])
        assert kbench.llms, "kaggle_benchmarks has no models: enable Kaggle model access for this notebook"
    JUDGE_DIR = WORK / "judgments"
    JUDGE_DIR.mkdir(exist_ok=True)
    pair_sources = list((REPO / "outputs" / "eval").glob("*_pairs.jsonl")) or list(Path("/kaggle/input").rglob("*_pairs.jsonl"))
    for p in pair_sources:
        shutil.copy(p, JUDGE_DIR / p.name)
    if PREVIOUS_JUDGMENTS:  # resume: restore judged/failure files from an earlier saved version
        for p in Path(PREVIOUS_JUDGMENTS).glob("*_judged*.jsonl"):
            shutil.copy(p, JUDGE_DIR / p.name)
    print(sorted(p.name for p in JUDGE_DIR.iterdir()))

In [6]:
# Judging runs in this kernel (not a subprocess) so it uses the task notebook's model-proxy credentials directly.
from alignment.judge import run_judging
from alignment.pairwise import summarize_judgments
from alignment.utils import write_json

RUBRICS = {"uf_dpo_vs_sft": "general", "hh_dpo_vs_sft": "safety", "persona_lora_vs_base": "persona"}
if RUN_PART_B_JUDGING:
    for name, rubric in RUBRICS.items():
        pairs = JUDGE_DIR / f"{name}_pairs.jsonl"
        if not pairs.exists():
            print("missing", pairs); continue
        judged = JUDGE_DIR / f"{name}_judged.jsonl"
        result = run_judging(pairs, judged, JUDGE_MODEL, rubric, batch_size=5,
                             sleep_seconds=0 if JUDGE_MODEL == "mock" else 5, max_retries=3, reuse_judgments=True)
        criteria = ["style_winner", "quality_winner"] if rubric == "persona" else ["winner"]
        summary = {c: summarize_judgments(pairs, judged, result["failures_path"], c) for c in criteria}
        write_json(JUDGE_DIR / f"{name}_judged_summary.json", summary)
        print(name, result, json.dumps({c: v["overall"] for c, v in summary.items()}, indent=1))

In [7]:
if RUN_PART_B_JUDGING:
    for p in sorted(JUDGE_DIR.glob("*_summary.json")):
        print(p.name, json.dumps({k: v["overall"] for k, v in json.load(open(p)).items()}, indent=1))
    with zipfile.ZipFile(WORK / "judgments.zip", "w", zipfile.ZIP_DEFLATED) as z:
        for p in JUDGE_DIR.iterdir():
            z.write(p, f"judgments/{p.name}")
    print("Saved", WORK / "judgments.zip", "- Save Version now, then copy *_judged*.jsonl into outputs/eval/ locally.")